# Evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os, pickle

import tensorflow as tf
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
os.makedirs('../outputs/figures', exist_ok=True)

train_df = pd.read_csv('../data/processed/train.csv')
test_df  = pd.read_csv('../data/processed/test.csv')
with open('../data/processed/scaler_y.pkl','rb') as f: scaler_y = pickle.load(f)
with open('../outputs/models/dummy.pkl','rb') as f: dummy = pickle.load(f)
with open('../outputs/models/linear_regression.pkl','rb') as f: lr = pickle.load(f)
nn = tf.keras.models.load_model('../outputs/models/best_nn.keras')

feat_cols = [c for c in train_df.columns if c not in ['loan_amnt_scaled','loan_amnt_raw']]
X_test    = test_df[feat_cols].values
y_test    = test_df['loan_amnt_raw'].values

y_pred_dm = scaler_y.inverse_transform(dummy.predict(X_test).reshape(-1,1)).flatten()
y_pred_lr = lr.predict(X_test)
y_pred_nn = scaler_y.inverse_transform(nn.predict(X_test, verbose=0).reshape(-1,1)).flatten()

print(f'Test set: {len(y_test):,} samples  |  Range: ${y_test.min():,}–${y_test.max():,}')

## Metrics

In [ ]:
def metrics(y_true, y_pred, name):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return {'Model': name, 'RMSE': f'${rmse:,.0f}', 'MAE': f'${mae:,.0f}',
            'R²': f'{r2:.4f}', 'MAPE': f'{mape:.1f}%'}

pd.DataFrame([
    metrics(y_test, y_pred_dm, 'Dummy (mean $9,589)'),
    metrics(y_test, y_pred_lr, 'Linear Regression'),
    metrics(y_test, y_pred_nn, 'Neural Network'),
])

## Actual vs Predicted

In [ ]:
np.random.seed(42)
sample = np.random.choice(len(y_test), 1500, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, y_pred, name, color in [
    (axes[0], y_pred_lr, 'Linear Regression', '#FF9800'),
    (axes[1], y_pred_nn, 'Neural Network',     '#1565C0'),
]:
    ax.scatter(y_test[sample]/1000, y_pred[sample]/1000, alpha=0.25, s=12, color=color)
    ax.plot([0,36],[0,36],'r--', lw=1.5, label='Perfect prediction')
    ax.set_title(f'{name}  R²={r2_score(y_test, y_pred):.3f}  MAE=${mean_absolute_error(y_test, y_pred):,.0f}', fontweight='bold')
    ax.set_xlabel('Actual ($k)'); ax.set_ylabel('Predicted ($k)')
    ax.set_xlim(0,36); ax.set_ylim(0,36); ax.legend()

plt.tight_layout()
plt.savefig('../outputs/figures/actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

## Accuracy Brackets

In [ ]:
errors_pct = np.abs(y_test - y_pred_nn) / y_test * 100
thresholds = [10, 20, 30, 50, 75]
within = [(errors_pct <= t).mean()*100 for t in thresholds]

for t, w in zip(thresholds, within):
    print(f'  Within ±{t:2d}%: {w:5.1f}%  ({int(w/100*len(y_test)):,} of {len(y_test):,})')

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar([f'±{t}%' for t in thresholds], within, color='#1565C0', edgecolor='white', alpha=0.85)
for bar, w in zip(bars, within):
    ax.text(bar.get_x()+bar.get_width()/2, w+0.5, f'{w:.0f}%', ha='center', fontweight='bold')
ax.set_ylabel('% of predictions'); ax.set_ylim(0, 80)
ax.set_title('Prediction accuracy brackets — Neural Network', fontweight='bold')
plt.tight_layout()
plt.savefig('../outputs/figures/accuracy_brackets.png', dpi=150, bbox_inches='tight')
plt.show()

## Residuals

In [ ]:
residuals = y_test - y_pred_nn

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].scatter(y_pred_nn[sample]/1000, residuals[sample]/1000, alpha=0.25, s=12, color='#C62828')
axes[0].axhline(0, color='black', lw=1.5, linestyle='--')
axes[0].set_xlabel('Predicted ($k)'); axes[0].set_ylabel('Residual ($k)')
axes[0].set_title('Residuals vs predicted — under-predicts large loans', fontweight='bold')

axes[1].hist(residuals/1000, bins=60, color='#1565C0', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='red', lw=1.5, linestyle='--')
axes[1].axvline(residuals.mean()/1000, color='orange', lw=2, linestyle='--',
                label=f'Mean = ${residuals.mean():,.0f}')
axes[1].set_xlabel('Residual ($k)'); axes[1].set_ylabel('Count')
axes[1].set_title(f'Distribution of residuals  std=${residuals.std():,.0f}', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('../outputs/figures/residuals.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean residual: ${residuals.mean():,.0f}')
print(f'Std residual:  ${residuals.std():,.0f}')